<img src="https://raw.githubusercontent.com/unslothai/unsloth/main/images/unsloth%20new%20logo.png" width="200"/>

# Unsloth × ExLlamaV3 (EXL3)
## Qwen3.5-9B at 6-bit - high-fidelity quantization + QLoRA

bitsandbytes tops out at 8-bit and 4-bit. EXL3 fills the gap with
**6-bit** (and 2/3/4-bit), giving near-lossless quality at a fraction of
the full-precision memory.

Here we quantize **`Qwen/Qwen3.5-9B`** to **6-bit** and QLoRA-fine-tune
it. 6-bit is a great pick when you want maximum quality retention and
have the VRAM to spare.

| | Value |
|---|---|
| Model | Qwen/Qwen3.5-9B (dense) |
| Quantization | EXL3 6-bit decoder / 8-bit head |

In [ ]:
# Install Unsloth with the EXL3 (ExLlamaV3) backend. Requires a CUDA GPU
# and a CUDA 12.4+ build of PyTorch.
%pip install -q "unsloth[exllama]"

### 1. Load & quantize to 6-bit

In [ ]:
import torch
from unsloth import FastLanguageModel
from unsloth.exllama import Exl3Config

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3.5-9B",
    max_seq_length = 2048,
    dtype = torch.bfloat16,
    # 6-bit decoder + 8-bit head: near-lossless, unavailable in bitsandbytes.
    load_in_exl3 = Exl3Config(bits = 6, head_bits = 8, calibrate = True),
)

### 2. Quick generation sanity check

In [ ]:
FastLanguageModel.for_inference(model)
messages = [{"role": "user", "content": "Explain gradient descent in two sentences."}]
ids = tokenizer.apply_chat_template(
    messages, add_generation_prompt = True, return_tensors = "pt",
).to("cuda")
out = model.generate(input_ids = ids, max_new_tokens = 96, do_sample = False)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens = True))

### 3. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 32, lora_dropout = 0.0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth", random_state = 3407,
)
model.print_trainable_parameters()

### Fine-tune with LoRA

A tiny slice of a dataset keeps this quick; swap in your own for real runs.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt

dataset = load_dataset("mlabonne/FineTome-100k", split = "train[:500]")
dataset = standardize_sharegpt(dataset)

def fmt(ex):
    return {
        "text": tokenizer.apply_chat_template(
            ex["conversations"], tokenize = False, add_generation_prompt = False,
        ),
    }
dataset = dataset.map(fmt)

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model, train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 60, learning_rate = 2e-4,
        logging_steps = 5, optim = "adamw_8bit", lr_scheduler_type = "cosine",
        seed = 3407, output_dir = "outputs", report_to = "none",
    ),
)
trainer.train()


Save the trained LoRA adapter; it reloads onto the 6-bit base.

In [ ]:
model.save_pretrained("Qwen3.5-9B-exl3-6bit-lora")
tokenizer.save_pretrained("Qwen3.5-9B-exl3-6bit-lora")
print("saved LoRA adapter")